# Preserve a .vmesh through USDC

Set the paths below. This works with every supported codec and needs OpenUSD, but no codec runtime.
The USDC stores compressed native state in O4D's custom schema; it is not a renderable mesh export.

In [ ]:
import hashlib
from pathlib import Path
import open4d
from open4d.codec import inspect_vmesh, pack_vmesh, unpack_vmesh

source = Path("motion.vmesh")
output = Path("output/native-usdc")

In [ ]:
info = inspect_vmesh(source)
print(info["codec"], info["frame_count"], "frames")

In [ ]:
with open4d.NativeSequence(source) as native:
    open4d.save(native, output / "native.usdc", overwrite=True)

In [ ]:
with open4d.load(output / "native.usdc") as native:
    restored = open4d.save(native, output / "restored.vmesh", overwrite=True)

with source.open("rb") as a, restored.open("rb") as b:
    assert hashlib.file_digest(a, "sha256").digest() == hashlib.file_digest(b, "sha256").digest()
print("Identical compressed bytes")

To inspect or migrate native files without decoding:

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as folder:
    native_files = unpack_vmesh(source, Path(folder) / "native")
    repacked = pack_vmesh(native_files, output / "repacked.vmesh", overwrite=True)
print(inspect_vmesh(repacked)["codec"])
